# Conjugate Priors for Cybersecurity Anomaly Detection

**Demonstrating Dirichlet-Categorical conjugate priors on the LANL authentication dataset**

This notebook accompanies the article "Understanding Conjugate Priors Through Real-World Data" and provides a complete implementation of Bayesian anomaly detection using conjugate priors.

## Dataset
- **LANL Comprehensive Multi-Source Cyber-Security Events Dataset**
- 1.648 billion authentication events over 58 days
- 749 red team attack events
- Available at: https://csr.lanl.gov/data/cyber1/

## What this notebook demonstrates:
1. **Dirichlet-Categorical conjugate priors** for categorical data
2. **Online Bayesian learning** with streaming data
3. **Temporal train/test splits** for realistic evaluation
4. **Computer-window labeling** strategy
5. **Performance evaluation** with ROC/PR curves


## Setup and Data Preparation

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import gzip
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score)
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("All libraries imported.")

In [ ]:
# File paths — works on Google Colab and locally
try:
    from google.colab import drive
    drive.mount('/content/drive')
    AUTH_FILE = '/content/drive/MyDrive/Lanl_login_data/auth.txt.gz'
    REDTEAM_FILE = '/content/drive/MyDrive/Lanl_login_data/redteam.txt.gz'
    PLOTS_DIR = '/content/drive/MyDrive/lanl_plots'
    print("Google Drive mounted.")
except ImportError:
    # Running locally — update paths to match your setup
    AUTH_FILE = './data/auth.txt.gz'
    REDTEAM_FILE = './data/redteam.txt.gz'
    PLOTS_DIR = './plots'
    print("Running locally.")

import os
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f"Auth data:    {AUTH_FILE}")
print(f"Red team data: {REDTEAM_FILE}")
print(f"Plots dir:    {PLOTS_DIR}")

## Mathematical Foundation: Dirichlet-Categorical Conjugate Priors

### The Model

For each computer $c$, we model authentication patterns using:

**Prior:** $\boldsymbol{\theta}^{(c)} \sim \text{Dir}(\alpha, \alpha, \ldots, \alpha)$

**Likelihood:** $\text{auth\_type} | \text{computer } c \sim \text{Categorical}(\boldsymbol{\theta}^{(c)})$

**Posterior:** $\boldsymbol{\theta}^{(c)} | \text{data} \sim \text{Dir}(\alpha + \boldsymbol{n}^{(c)})$

### The Update Rule

The beautiful simplicity of conjugate priors:

$$P(\text{category } k | \text{data}) = \frac{\alpha + n_k}{\alpha_0 + N}$$

**Anomaly Score:** $-\log P(\text{category } k | \text{data})$

### Why α = 1 (Uniform Prior)?

- **No bias:** Equal initial belief for all authentication types
- **Minimal influence:** Lets data drive the learning
- **Smooth handling:** Prevents zero probabilities for unseen categories

## Implementation

In [4]:
class DirichletCategorical:
    """
    Bayesian categorical model using Dirichlet-Categorical conjugate priors.
    
    This implements the mathematical framework:
    θ ~ Dir(α)
    x | θ ~ Categorical(θ)  
    θ | data ~ Dir(α + counts)
    """
    
    def __init__(self, alpha_prior=1.0):
        """
        Initialize with symmetric Dirichlet prior.
        
        Parameters:
        -----------
        alpha_prior : float
            Prior pseudo-count for each category (α parameter)
        """
        self.counts = defaultdict(int)  # Observed category counts n_k
        self.alpha_prior = alpha_prior  # Prior parameter α
        self.total = 0                  # Total observations N
        
    def update(self, observations):
        """
        Online Bayesian update: increment sufficient statistics.
        
        This implements the conjugate prior update:
        Dir(α) → Dir(α + n) where n are new counts
        """
        for obs in observations:
            self.counts[obs] += 1
            self.total += 1
    
    def anomaly_score(self, category):
        """
        Compute surprisal: -log(probability)
        
        Formula: -log((α + n_k) / (α_0 + N))
        Higher score = more anomalous
        """
        # Posterior parameters
        alpha_posterior = self.counts.get(category, 0) + self.alpha_prior
        total_alpha = self.total + len(self.counts) * self.alpha_prior
        
        # For unseen categories, add α to denominator
        if category not in self.counts:
            total_alpha += self.alpha_prior
        
        # Posterior predictive probability
        probability = alpha_posterior / total_alpha
        
        # Anomaly score
        return -np.log(probability + 1e-10)  # Numerical stability
    
    def get_statistics(self):
        """Get model statistics for analysis."""
        return {
            'total_observations': self.total,
            'unique_categories': len(self.counts),
            'most_common': max(self.counts.items(), key=lambda x: x[1]) if self.counts else None,
            'entropy': self._compute_entropy()
        }
    
    def _compute_entropy(self):
        """Compute the entropy of the learned distribution."""
        if not self.counts:
            return 0.0
            
        total = self.total + len(self.counts) * self.alpha_prior
        entropy = 0.0
        
        for count in self.counts.values():
            prob = (count + self.alpha_prior) / total
            entropy -= prob * np.log(prob + 1e-10)
            
        return entropy

print("✅ DirichletCategorical class defined")

✅ DirichletCategorical class defined


In [ ]:
class EnterpriseAuthDetector:
    """
    Multi-signal Bayesian anomaly detector for enterprise authentication events.
    Models two categorical distributions per computer:
    1. Authentication type patterns
    2. Source user access patterns
    """

    def __init__(self, alpha_prior=1.0):
        self.computer_models = {}
        self.global_models = {
            'auth_types': DirichletCategorical(alpha_prior),
            'users': DirichletCategorical(alpha_prior)
        }
        self.alpha_prior = alpha_prior
        self.training_stats = {}

    def train_on_events(self, events_iterator, max_timestamp=None):
        """Online Bayesian update: process each event, increment sufficient statistics."""
        events_processed = 0

        print("Training Bayesian models on authentication patterns...")
        for event in tqdm(events_iterator, desc="Learning normal behaviors",
                          miniters=50000, unit_scale=True):
            if max_timestamp and event['timestamp'] >= max_timestamp:
                print(f"\nStopped at timestamp {event['timestamp']}")
                break
            if event['success'] != 'Success':
                continue

            computer  = event['dest_computer']
            auth_type = event['auth_type']
            user      = event['source_user']

            if computer not in self.computer_models:
                self.computer_models[computer] = {
                    'auth_types': DirichletCategorical(self.alpha_prior),
                    'users':      DirichletCategorical(self.alpha_prior)
                }

            self.computer_models[computer]['auth_types'].update([auth_type])
            self.computer_models[computer]['users'].update([user])
            self.global_models['auth_types'].update([auth_type])
            self.global_models['users'].update([user])
            events_processed += 1

        self.training_stats = {
            'events_processed':   events_processed,
            'computers_modeled':  len(self.computer_models),
            'global_auth_types':  len(self.global_models['auth_types'].counts),
            'global_users':       len(self.global_models['users'].counts)
        }
        print(f"Training complete: {events_processed:,} events, "
              f"{len(self.computer_models):,} computers modeled.")

    def score_event(self, event):
        """Compute anomaly score: -log(posterior predictive probability)."""
        computer  = event['dest_computer']
        auth_type = event['auth_type']
        user      = event['source_user']

        if computer in self.computer_models:
            auth_score = self.computer_models[computer]['auth_types'].anomaly_score(auth_type)
            user_score = self.computer_models[computer]['users'].anomaly_score(user)
            model_type = 'computer_specific'
        else:
            auth_score = self.global_models['auth_types'].anomaly_score(auth_type)
            user_score = self.global_models['users'].anomaly_score(user)
            model_type = 'global_fallback'

        return {
            'auth_score':     auth_score,
            'user_score':     user_score,
            'combined_score': (auth_score + user_score) / 2,
            'model_type':     model_type
        }

    def analyze_patterns(self):
        """Summarize what the detector learned."""
        global_auth_stats = self.global_models['auth_types'].get_statistics()
        global_user_stats = self.global_models['users'].get_statistics()

        computer_specializations = {}
        for computer, models in list(self.computer_models.items())[:10]:
            auth_stats = models['auth_types'].get_statistics()
            user_stats = models['users'].get_statistics()
            computer_specializations[computer] = {
                'auth_entropy':      auth_stats['entropy'],
                'user_entropy':      user_stats['entropy'],
                'total_auth_events': auth_stats['total_observations'],
                'most_common_auth':  auth_stats['most_common'],
                'unique_users':      user_stats['unique_categories']
            }

        print(f"\nLEARNED PATTERNS:")
        print(f"  Computers modeled: {len(self.computer_models):,}")
        print(f"  Global auth types: {global_auth_stats['unique_categories']:,}")
        print(f"  Global users:      {global_user_stats['unique_categories']:,}")

        return {
            'global_patterns':          {'auth_types': global_auth_stats, 'users': global_user_stats},
            'computer_specializations': computer_specializations,
            'summary':                  {
                'total_computers':   len(self.computer_models),
                'global_auth_types': global_auth_stats['unique_categories'],
                'global_users':      global_user_stats['unique_categories']
            }
        }

print("EnterpriseAuthDetector defined.")

In [ ]:
def load_lanl_dataset(auth_file, redteam_file):
    """Load LANL dataset and return red-team DataFrame + auth event iterator."""

    print("Loading LANL dataset...")

    red_team_events = []
    with gzip.open(redteam_file, 'rt') as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 4:
                red_team_events.append({
                    'timestamp':       int(parts[0]),
                    'user':            parts[1],
                    'source_computer': parts[2],
                    'dest_computer':   parts[3]
                })

    red_team_df = pd.DataFrame(red_team_events)
    print(f"Red team: {len(red_team_df):,} attack events, "
          f"{red_team_df.dest_computer.nunique()} compromised computers")

    def _normalize_auth_type(auth_type):
        """Collapse the many truncated MICROSOFT_AUTHENTICATION_PACKAGE variants."""
        if auth_type.startswith('MICROSOFT_AUTHENTICATION_PACKAGE'):
            return 'MSAUTHPKG'
        return auth_type

    def auth_event_iterator():
        with gzip.open(auth_file, 'rt') as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) == 9:
                    yield {
                        'timestamp':        int(parts[0]),
                        'source_user':      parts[1],
                        'dest_user':        parts[2],
                        'source_computer':  parts[3],
                        'dest_computer':    parts[4],
                        'auth_type':        _normalize_auth_type(parts[5]),
                        'logon_type':       parts[6],
                        'auth_orientation': parts[7],
                        'success':          parts[8]
                    }

    return red_team_df, auth_event_iterator


def create_temporal_split(red_team_df):
    """Train on pre-attack data only — realistic deployment simulation."""
    training_cutoff = red_team_df.timestamp.min()
    compromised_computers = set(red_team_df.dest_computer.unique())

    split_info = {
        'training_cutoff':       training_cutoff,
        'test_start':            red_team_df.timestamp.min(),
        'test_end':              red_team_df.timestamp.max() + 86400,
        'compromised_computers': compromised_computers,
        'attack_duration_days':  (red_team_df.timestamp.max() - red_team_df.timestamp.min()) / 86400
    }

    print(f"Training cutoff:  {training_cutoff}")
    print(f"Attack duration:  {split_info['attack_duration_days']:.1f} days")
    print(f"Compromised hosts:{len(compromised_computers)}")
    return split_info

print("Data loading functions defined.")

In [ ]:
def collect_test_events(auth_iterator, split_info, max_events=200000):
    """Collect labelled test events from the attack window."""
    test_events = []
    test_start = split_info['test_start']
    test_end   = split_info['test_end']

    print("Collecting test events from attack period...")
    for event in tqdm(auth_iterator, desc="Collecting test data",
                      miniters=10000, unit_scale=True):
        if not (test_start <= event['timestamp'] <= test_end):
            continue
        if event['success'] != 'Success':
            continue

        is_attack = event['dest_computer'] in split_info['compromised_computers']
        test_events.append({
            'timestamp':    event['timestamp'],
            'dest_computer':event['dest_computer'],
            'auth_type':    event['auth_type'],
            'source_user':  event['source_user'],
            'is_attack':    is_attack
        })

        if len(test_events) >= max_events:
            break

    print(f"Collected {len(test_events):,} test events")
    return test_events


def balance_evaluation_set(test_events, target_ratio=100):
    """Keep all attack events, sample normals to achieve target_ratio."""
    df = pd.DataFrame(test_events)
    attacks = df[df.is_attack]
    normals = df[~df.is_attack]

    if len(attacks) == 0:
        raise ValueError("No attack events in test set — check temporal split.")

    n_normal = min(len(normals), len(attacks) * target_ratio)
    balanced = pd.concat([attacks, normals.sample(n_normal, random_state=42)]).reset_index(drop=True)

    print(f"Evaluation set: {len(attacks):,} attacks + {n_normal:,} normal "
          f"(1:{n_normal // len(attacks)} ratio)")
    return balanced


def evaluate_performance(evaluation_set):
    """Compute AUC-ROC, average precision, Precision@K, and score separation stats."""
    y_true   = evaluation_set.is_attack.astype(int)
    y_scores = evaluation_set.combined_score

    auc_roc = roc_auc_score(y_true, y_scores)
    avg_precision = average_precision_score(y_true, y_scores)   # replaces np.trapz (which gave wrong sign)

    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    fpr, tpr, roc_thresh = roc_curve(y_true, y_scores)

    sorted_df = evaluation_set.sort_values('combined_score', ascending=False)
    precision_at_k = {}
    for k in [10, 25, 50, 100, 200, 500]:
        if k <= len(sorted_df):
            precision_at_k[f'P@{k}'] = sorted_df.head(k).is_attack.sum() / k

    attack_scores = evaluation_set[evaluation_set.is_attack].combined_score
    normal_scores = evaluation_set[~evaluation_set.is_attack].combined_score
    stat, p_value = mannwhitneyu(attack_scores, normal_scores, alternative='greater')

    pooled_std = np.sqrt((attack_scores.var() + normal_scores.var()) / 2)
    cohens_d   = (attack_scores.mean() - normal_scores.mean()) / pooled_std

    results = {
        'auc_roc':        auc_roc,
        'avg_precision':  avg_precision,
        'precision_at_k': precision_at_k,
        'score_separation': {
            'attack_mean':   attack_scores.mean(),
            'normal_mean':   normal_scores.mean(),
            'cohens_d':      cohens_d,
            'mannwhitney_p': p_value
        },
        'roc_curve': (fpr, tpr, roc_thresh),
        'pr_curve':  (precision, recall)
    }

    print(f"\nPERFORMANCE:")
    print(f"  AUC-ROC:           {auc_roc:.4f}")
    print(f"  Average Precision: {avg_precision:.4f}")
    print(f"  Cohen's d:         {cohens_d:.3f}")
    print(f"  Attack score mean: {attack_scores.mean():.2f}")
    print(f"  Normal score mean: {normal_scores.mean():.2f}")
    print(f"  Mann-Whitney p:    {p_value:.2e}")
    print(f"\nPrecision@K:")
    for k, v in precision_at_k.items():
        print(f"  {k}: {v:.1%}")

    return results


def create_evaluation_plots(evaluation_set, results, plots_dir='.'):
    """Generate and save the four core evaluation figures."""
    os.makedirs(plots_dir, exist_ok=True)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # ROC Curve
    fpr, tpr, _ = results['roc_curve']
    axes[0,0].plot(fpr, tpr, linewidth=2.5, color='steelblue',
                   label=f"AUC = {results['auc_roc']:.4f}")
    axes[0,0].plot([0,1],[0,1], 'k--', alpha=0.4, label='Random')
    axes[0,0].set_xlabel('False Positive Rate'); axes[0,0].set_ylabel('True Positive Rate')
    axes[0,0].set_title('ROC Curve'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

    # Precision-Recall Curve
    precision, recall = results['pr_curve']
    axes[0,1].plot(recall, precision, linewidth=2.5, color='coral',
                   label=f"Avg Precision = {results['avg_precision']:.4f}")
    baseline = evaluation_set.is_attack.mean()
    axes[0,1].axhline(baseline, color='k', linestyle='--', alpha=0.4, label='Random baseline')
    axes[0,1].set_xlabel('Recall'); axes[0,1].set_ylabel('Precision')
    axes[0,1].set_title('Precision-Recall Curve'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

    # Score Distributions
    attack_scores = evaluation_set[evaluation_set.is_attack].combined_score
    normal_scores = evaluation_set[~evaluation_set.is_attack].combined_score
    axes[1,0].hist(normal_scores, bins=60, alpha=0.6, density=True, color='steelblue', label='Normal')
    axes[1,0].hist(attack_scores, bins=60, alpha=0.6, density=True, color='coral',    label='Attack')
    axes[1,0].axvline(normal_scores.mean(), color='steelblue', linestyle='--', linewidth=1.5)
    axes[1,0].axvline(attack_scores.mean(), color='coral',     linestyle='--', linewidth=1.5)
    axes[1,0].set_xlabel('Anomaly Score'); axes[1,0].set_ylabel('Density')
    axes[1,0].set_title(f"Score Distributions  (Cohen's d = {results['score_separation']['cohens_d']:.2f})")
    axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

    # Precision@K
    k_vals = [int(k.split('@')[1]) for k in results['precision_at_k']]
    p_vals = list(results['precision_at_k'].values())
    axes[1,1].plot(k_vals, p_vals, 'o-', linewidth=2.5, markersize=7, color='seagreen')
    axes[1,1].set_xlabel('K (top-K alerts)'); axes[1,1].set_ylabel('Precision')
    axes[1,1].set_title('Precision@K — Operational Performance')
    axes[1,1].set_ylim(0, 1); axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(plots_dir, 'evaluation_plots.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {path}")
    return fig

print("Evaluation functions defined.")

In [ ]:
def run_anomaly_detection_pipeline(auth_file=AUTH_FILE, redteam_file=REDTEAM_FILE,
                                    plots_dir=PLOTS_DIR):
    """End-to-end pipeline: load → train → score → evaluate → plot."""

    red_team_df, auth_iterator = load_lanl_dataset(auth_file, redteam_file)
    split_info = create_temporal_split(red_team_df)

    detector = EnterpriseAuthDetector(alpha_prior=1.0)
    detector.train_on_events(auth_iterator(), max_timestamp=split_info['training_cutoff'])
    pattern_analysis = detector.analyze_patterns()

    test_events = collect_test_events(auth_iterator(), split_info)

    print("Scoring test events...")
    scored_events = []
    for event in tqdm(test_events, desc="Computing scores", miniters=10000):
        scored_events.append({**event, **detector.score_event(event)})

    evaluation_set = balance_evaluation_set(scored_events, target_ratio=100)
    results        = evaluate_performance(evaluation_set)
    fig            = create_evaluation_plots(evaluation_set, results, plots_dir=plots_dir)

    # Save auth-type distribution plot separately
    plot_auth_distribution(detector, plots_dir=plots_dir)

    return {
        'detector':        detector,
        'evaluation_set':  evaluation_set,
        'results':         results,
        'pattern_analysis':pattern_analysis,
        'figure':          fig
    }


def plot_auth_distribution(detector, plots_dir='.'):
    """Horizontal bar chart of global authentication type frequencies."""
    auth_counts = detector.global_models['auth_types'].counts
    total = sum(auth_counts.values())

    sorted_items = sorted(auth_counts.items(), key=lambda x: x[1], reverse=True)
    labels = [item[0] for item in sorted_items]
    values = [item[1] / total * 100 for item in sorted_items]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(labels[::-1], values[::-1], color='steelblue', edgecolor='white')
    ax.set_xlabel('Percentage of Events (%)')
    ax.set_title('Global Authentication Type Distribution\n(LANL Dataset — 29.4M training events)')
    ax.grid(True, axis='x', alpha=0.3)
    for bar, val in zip(bars, values[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    path = os.path.join(plots_dir, 'auth_distribution.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {path}")

print("Pipeline functions defined.")

## Execute the Complete Pipeline

This section runs the full anomaly detection pipeline. Make sure you have:

1. ✅ Mounted Google Drive
2. ✅ Updated the file paths to point to your LANL dataset
3. ✅ Sufficient RAM (recommended: High-RAM runtime)

**Expected runtime:** 15-20 minutes on standard Colab

In [ ]:
pipeline_results = run_anomaly_detection_pipeline()
print("\nPipeline complete. Plots saved to:", PLOTS_DIR)

## Analysis of Results

Let's examine what our Bayesian models learned and how well they performed.

In [10]:
# Extract results for detailed analysis
detector = pipeline_results['detector']
evaluation_set = pipeline_results['evaluation_set']
results = pipeline_results['results']
pattern_analysis = pipeline_results['pattern_analysis']

print("=" * 60)
print("📊 DETAILED ANALYSIS OF BAYESIAN LEARNING")
print("=" * 60)

# Training Statistics
print(f"\n🧠 TRAINING PHASE:")
print(f"   Events processed: {detector.training_stats['events_processed']:,}")
print(f"   Computers modeled: {detector.training_stats['computers_modeled']:,}")
print(f"   Unique auth types: {detector.training_stats['global_auth_types']:,}")
print(f"   Unique users: {detector.training_stats['global_users']:,}")

# Global Authentication Patterns
print(f"\n🌐 GLOBAL AUTHENTICATION PATTERNS:")
auth_counts = detector.global_models['auth_types'].counts
total_auth = sum(auth_counts.values())

for auth_type, count in sorted(auth_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_auth) * 100
    anomaly_score = detector.global_models['auth_types'].anomaly_score(auth_type)
    print(f"   {auth_type:15s}: {count:8,} events ({percentage:5.1f}%) - Score: {anomaly_score:.2f}")

# Performance Summary
print(f"\n🎯 PERFORMANCE SUMMARY:")
print(f"   AUC-ROC: {results['auc_roc']:.4f}")
print(f"   AUC-PR:  {results['auc_pr']:.4f}")
print(f"   Cohen's d: {results['score_separation']['cohens_d']:.3f}")

# Score Separation
attack_mean = results['score_separation']['attack_mean']
normal_mean = results['score_separation']['normal_mean']
print(f"\n📊 SCORE SEPARATION:")
print(f"   Attack events: μ = {attack_mean:.2f}")
print(f"   Normal events: μ = {normal_mean:.2f}")
print(f"   Difference: {attack_mean - normal_mean:.2f} (higher = better separation)")

# Top Computer Specializations
print(f"\n🖥️  COMPUTER SPECIALIZATION EXAMPLES:")
for i, (computer, info) in enumerate(list(pattern_analysis['computer_specializations'].items())[:5]):
    if info['most_common_auth']:
        auth_type, count = info['most_common_auth']
        total = info['total_auth_events']
        specialization = (count / total) * 100 if total > 0 else 0
        print(f"   {computer}: {specialization:.1f}% {auth_type} ({total:,} total events)")

print(f"\n✨ Mathematical elegance achieved: Online Bayesian learning with conjugate priors!")

📊 DETAILED ANALYSIS OF BAYESIAN LEARNING

🧠 TRAINING PHASE:
   Events processed: 29,425,112
   Computers modeled: 10,413
   Unique auth types: 17
   Unique users: 34,897

🌐 GLOBAL AUTHENTICATION PATTERNS:
   ?              : 17,004,222 events ( 57.8%) - Score: 0.55
   Kerberos       : 10,367,997 events ( 35.2%) - Score: 1.04
   NTLM           : 1,431,374 events (  4.9%) - Score: 3.02
   Negotiate      :  604,153 events (  2.1%) - Score: 3.89
   MICROSOFT_AUTHENTICATION_PACKAGE_V1_0:   11,850 events (  0.0%) - Score: 7.82
   MICROSOFT_AUTHENTICATION_PACKAG:    1,469 events (  0.0%) - Score: 9.90
   MICROSOFT_AUTHENTICATION_PACKAGE_V1_:    1,046 events (  0.0%) - Score: 10.24
   MICROSOFT_AUTHENTICATION_PACKAGE_V1:      848 events (  0.0%) - Score: 10.45
   MICROSOFT_AUTHENTICATION_PACK:      787 events (  0.0%) - Score: 10.53
   MICROSOFT_AUTHENTICATION_PACKAGE:      653 events (  0.0%) - Score: 10.71
   MICROSOFT_AUTHENTICATION_PACKA:      273 events (  0.0%) - Score: 11.58
   MICROSOF

## Effect of the Prior Parameter α

How sensitive is the model to the choice of α? We can answer this analytically — no re-training needed.

In [ ]:
## Alpha sensitivity: mathematical demonstration (no re-training needed)
#
# For a category observed n_k times out of N total, with K known categories,
# the posterior predictive probability is:
#
#   P(x = k | data) = (alpha + n_k) / (K*alpha + N)
#
# We vary alpha and show how the anomaly score (-log P) changes.

import numpy as np
import matplotlib.pyplot as plt

alphas       = [0.01, 0.1, 1.0, 5.0, 10.0]
colors       = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
N            = 1000   # total observations
K            = 6      # number of known auth-type categories

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: anomaly score vs observation count for different α
obs_counts = np.arange(0, 101)
for alpha, color in zip(alphas, colors):
    scores = -np.log((alpha + obs_counts) / (K * alpha + N))
    axes[0].plot(obs_counts, scores, linewidth=2, color=color, label=f'α = {alpha}')

axes[0].set_xlabel('Times category observed (n_k)')
axes[0].set_ylabel('Anomaly Score  (−log P)')
axes[0].set_title('Anomaly Score vs Observation Count\n(N=1000 total events, K=6 categories)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axvline(0, color='gray', linestyle=':', linewidth=1)

# Right: score for unseen category (n_k = 0) across different N
N_range = np.logspace(1, 5, 300)
for alpha, color in zip(alphas, colors):
    scores_unseen = -np.log(alpha / (K * alpha + N_range))
    axes[1].semilogx(N_range, scores_unseen, linewidth=2, color=color, label=f'α = {alpha}')

axes[1].set_xlabel('Training set size N (log scale)')
axes[1].set_ylabel('Anomaly Score for Unseen Category')
axes[1].set_title('Sensitivity to α: Score for Never-Seen Auth Type\n(higher = more anomalous)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axvline(29_425_112, color='black', linestyle='--', alpha=0.6, label='LANL N')
axes[1].text(29_425_112 * 1.1, axes[1].get_ylim()[0] + 0.5, 'LANL N', fontsize=8)

plt.suptitle('Effect of Prior Strength α on Anomaly Scoring', fontweight='bold', fontsize=13)
plt.tight_layout()
path = os.path.join(PLOTS_DIR, 'alpha_sensitivity.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {path}")

# Print the table shown in the article
print("\nScore for a category seen 3 times globally (n_k=3, N=29M, K=6):")
print(f"{'α':>6} | {'P(category)':>14} | {'Anomaly Score':>14}")
print("-" * 40)
for alpha in alphas:
    N_lanl = 29_425_112
    p = (alpha + 3) / (K * alpha + N_lanl)
    score = -np.log(p)
    print(f"{alpha:>6} | {p:>14.3e} | {score:>14.2f}")

## Save Results

Save the evaluation results and trained model for further analysis.

In [ ]:
import json

evaluation_set = pipeline_results['evaluation_set']
results        = pipeline_results['results']
detector       = pipeline_results['detector']

evaluation_set.to_csv(os.path.join(PLOTS_DIR, 'lanl_evaluation_results.csv'), index=False)

metrics = {
    'auc_roc':          results['auc_roc'],
    'avg_precision':    results['avg_precision'],
    'precision_at_k':   results['precision_at_k'],
    'score_separation': results['score_separation'],
    'training_stats':   detector.training_stats
}
# numpy floats → Python floats for JSON
metrics = json.loads(json.dumps(metrics, default=float))

with open(os.path.join(PLOTS_DIR, 'lanl_performance_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print("Results saved to:", PLOTS_DIR)

## Conclusion

This notebook has demonstrated the mathematical elegance and practical power of **Dirichlet-Categorical conjugate priors** for cybersecurity anomaly detection.

### Key Achievements:

🧮 **Mathematical Elegance**: Posterior updates through simple addition  
⚡ **Computational Efficiency**: Linear time training, constant time inference  
🔍 **Interpretable Results**: Clear probabilistic meaning of anomaly scores  
📊 **Strong Performance**: Significant separation between attack and normal events  
🎯 **Practical Deployment**: No hyperparameter tuning required  

### The Core Insight:

**Complex Bayesian Inference → Simple Arithmetic**

When your data structure aligns with conjugate prior assumptions, you get both theoretical rigor and practical performance.

### Next Steps:

- Experiment with different α values
- Extend to additional categorical features
- Apply to your own categorical anomaly detection problems
- Explore hierarchical Dirichlet models

---

**"Sometimes the most sophisticated approach is also the most mathematically principled one."**
